In [1]:
import subprocess, sys, torch

tv = torch.__version__.split("+")[0]
cu = "cu" + torch.version.cuda.replace(".", "")
url = f"https://data.pyg.org/whl/torch-{tv}+{cu}.html"

subprocess.run([sys.executable,"-m","pip","install",
    "pyg_lib","torch_scatter","torch_sparse",
    "torch_cluster","torch_spline_conv","-f",url,"-q"], check=False)
subprocess.run([sys.executable,"-m","pip","install","torch_geometric","-q"])
print("✔ PyG installed")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 42.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 124.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 116.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 112.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 65.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.2 MB/s eta 0:00:00
✔ PyG installed


In [2]:
import shutil, os, sys

CODE = "/kaggle/input/datasets/jarinproma/pgcn-eeg-code"
for f in os.listdir(CODE):
    if f.endswith(".py"):
        shutil.copy(f"{CODE}/{f}", f"/kaggle/working/{f}")
        print(f"  Copied: {f}")
sys.path.insert(0, "/kaggle/working")
print("✔ Done")


  Copied: kaggle_notebooks.py
  Copied: preprocess_eeg.py
  Copied: pgcn_model.py
  Copied: pgcn_experiment.py
  Copied: preprocess_fusion.py
  Copied: pgcn_statistics_plots.py
  Copied: pgcn_train.py
  Copied: pgcn_core.py
✔ Done


In [3]:
import os, json

candidates = [
    "/kaggle/input/datasets/jarinproma/pgcn-deap-preprocessed",
    "/kaggle/input/datasets/jerinanan/pgcn-deap-preprocessed",
    "/kaggle/input/datasets/jarinproma/pgcn-deap-preprocessed",
]

DATA = None
for path in candidates:
    if os.path.exists(path):
        subs = [d for d in os.listdir(path) if d.startswith("s")]
        if subs and os.path.exists(f"{path}/{subs[0]}/meta.json"):
            DATA = path
            print(f"✔ Correct path: {DATA}")
            print(f"  Subjects: {len(subs)}")
            break

if DATA is None:
    print("✗ Path not found — check dataset name")


✔ Correct path: /kaggle/input/datasets/jarinproma/pgcn-deap-preprocessed
  Subjects: 32


In [4]:
BATCH = [f"s{i:02d}" for i in range(27, 33)]

from pgcn_experiment import run_experiment, print_summary
import pgcn_experiment as E

original_get = E.get_subjects
def get_batch(data_dir, fusion):
    return [s for s in original_get(data_dir, fusion) if s in BATCH]
E.get_subjects = get_batch

store = run_experiment(
    data_dir=DATA, out_dir="/kaggle/working/results_eeg/loso_flat_gcn",
    mode="loso", fusion="none", ablation="flat_gcn", force=False,
)
print_summary(store, out_dir="/kaggle/working/results_eeg/loso_flat_gcn/aggregate")
E.get_subjects = original_get
print("✔ A1 Flat GCN s11-s21 done!")


  mode=loso  fusion=none  ablation=flat_gcn
  data_dir : /kaggle/input/datasets/jarinproma/pgcn-deap-preprocessed
  out_dir  : /kaggle/working/results_eeg/loso_flat_gcn

  loso  fusion=none  ablation=flat_gcn  (6 test folds)

───────────────────────────────────────────────────────
  fold test=s27  train=31
  DEAPDataset [train]   25172 windows  (31 subjects)
  DEAPDataset [val  ]    5394 windows  (31 subjects)
  DEAPDataset [all  ]    1160 windows  (1 subjects)
  ep  1/100  tr=5.750  val=0.615  aro=0.714  mean=0.706
  ep 10/100  tr=5.299  val=0.644  aro=0.723  mean=0.718
  ep 20/100  tr=5.165  val=0.620  aro=0.707  mean=0.702
  Early stop ep22  best=0.7330
  TEST  val=0.7641  aro=0.7965  dom=0.9873  lik=0.9189  quad=0.1959

───────────────────────────────────────────────────────
  fold test=s28  train=31
  DEAPDataset [train]   25172 windows  (31 subjects)
  DEAPDataset [val  ]    5394 windows  (31 subjects)
  DEAPDataset [all  ]    1160 windows  (1 subjects)
  ep  1/100  tr=5.868  val

In [5]:
import shutil, os

shutil.make_archive(
    "/kaggle/working/loso_flat_gcn_s27_s32",
    "zip",
    "/kaggle/working",
    "results_eeg"
)
print(f"✔ {os.path.getsize('/kaggle/working/loso_flat_gcn_s27_s32.zip')/1e6:.1f} MB")

✔ 1.1 MB
